# Literature review: a question-to-evidence matrix

This notebook separates four operations that are often conflated: **retrieval** finds relevant passages; **summarisation** condenses a document; **structured extraction** places evidence into fixed fields; and **synthesis** compares those fields across papers. The default path uses local PDF text and tracked cached LLM tables, so it makes no network or API call.

**Modular mapping:** research question + PDFs → retrieved passages with page locations → provisional structured fields → comparison matrix.

**Prerequisites:** pandas tables, basic PDF text extraction, and critical reading of source passages.

## 1. Retrieve local evidence with its page location

The simple lexical retriever below is deliberately transparent for teaching. A semantic FAISS retriever is available as an optional live/embedding path through `llm_utils.build_pdf_retriever`, but it may download a model and is not part of the default smoke test.

In [ ]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display
from pypdf import PdfReader

from ccai9012 import llm_utils
from ccai9012.paths import STARTER_KITS_DIR, ensure_output_dir

EXAMPLE_ROOT = STARTER_KITS_DIR / "2_llm_structure_output" / "lit_review"
PDF_DIR = EXAMPLE_ROOT / "data"
OUTPUT_DIR = ensure_output_dir(EXAMPLE_ROOT)
pdf_paths = sorted(PDF_DIR.glob("*.pdf"))
assert pdf_paths

def extract_pages(pdf_path):
    reader = PdfReader(str(pdf_path))
    return [{"page": index + 1, "text": page.extract_text() or ""} for index, page in enumerate(reader.pages)]

def retrieve_passages(pages, query, top_k=2):
    terms = [term.lower() for term in re.findall(r"[A-Za-z]+", query) if len(term) > 2]
    scored = []
    for page in pages:
        text = page["text"].replace("\n", " ").strip()
        score = sum(text.lower().count(term) for term in terms)
        if text:
            scored.append((score, page["page"], text))
    return [
        {"page": page, "score": score, "text": text}
        for score, page, text in sorted(scored, reverse=True)[:top_k]
    ]

page_cache = {path.name: extract_pages(path) for path in pdf_paths}
evidence = retrieve_passages(page_cache[pdf_paths[0].name], "model methodology results", top_k=2)
print(f"pdfs={len(pdf_paths)}; pages_first_pdf={len(page_cache[pdf_paths[0].name])}")
for item in evidence:
    print(f"page={item['page']}; score={item['score']}; passage={item['text'][:240]}...")

## 2. Keep source, field, and uncertainty attached

The cached tables are provisional LLM notes from the original starter kit. We parse them separately from retrieval, retain the PDF filename, look up a supporting page with the local retriever, and write `N/A` when a field is missing. A cached answer is a reading aid—not evidence that removes the need to reopen the paper.

![A research question retrieves passages, which become an evidence matrix with paper, page, field, value, and uncertainty before synthesis.](../../../docs/figs/literature_evidence_matrix.svg)

In [ ]:
cached_path = EXAMPLE_ROOT / "output/multiple_comparison.csv"
cached = pd.read_csv(cached_path)
expected_fields = ["Problem", "Research Gap", "Methodology", "Key Results"]
matrix_rows = []
structured_rows = []

for _, cached_row in cached.iterrows():
    paper = str(cached_row["pdf_path"])
    parsed = llm_utils.parse_markdown_table(str(cached_row["extracted_text"]))
    for field in expected_fields:
        value = "N/A"
        if field in parsed.columns and not parsed.empty and pd.notna(parsed.iloc[0][field]):
            value = str(parsed.iloc[0][field]).strip() or "N/A"
        passages = retrieve_passages(page_cache.get(paper, []), field, top_k=1)
        page = passages[0]["page"] if passages else "N/A"
        passage = passages[0]["text"][:320] if passages else "N/A"
        matrix_rows.append({
            "paper": paper, "extracted_field": field, "value": value,
            "retrieved_passage": passage, "source_location": f"PDF page {page}",
            "uncertainty": "Provisional cached LLM note; verify against the original PDF.",
        })
        structured_rows.append({"paper": paper, field: value})

evidence_matrix = pd.DataFrame(matrix_rows)
evidence_matrix.to_csv(OUTPUT_DIR / "offline_evidence_matrix.csv", index=False)
assert {"paper", "extracted_field", "value", "retrieved_passage", "source_location", "uncertainty"} <= set(evidence_matrix.columns)
display(evidence_matrix.head(8))
print(f"evidence_rows={len(evidence_matrix)}; papers={evidence_matrix['paper'].nunique()}")

## 3. Compare fields without hiding missing evidence

The matrix is long-form so one paper can have one row per field and still retain a passage and location. Count `N/A` explicitly; do not turn a retrieval miss into a claim that the paper has no evidence. The following table is a compact comparison view for discussion.

In [ ]:
comparison = evidence_matrix.pivot(index="paper", columns="extracted_field", values="value").reset_index()
comparison = comparison.reindex(columns=["paper"] + expected_fields)
comparison.to_csv(OUTPUT_DIR / "offline_literature_comparison.csv", index=False)
na_by_field = evidence_matrix.assign(is_na=evidence_matrix["value"].eq("N/A")).groupby("extracted_field")["is_na"].sum()
display(comparison)
print("N/A counts by field:")
display(na_by_field.to_frame("missing_rows"))

## 4. Optional semantic/LLM branch

To run the original LLM-assisted path, set `RUN_LIVE_LLM = True`, build a retriever with `llm_utils.build_pdf_retriever`, and pass its retrieved context to `llm_utils.run_qa_chain`. Record the model, date, prompt, source pages, and cost boundary. The default notebook intentionally stops at the cached/local path.

In [ ]:
RUN_LIVE_LLM = False
if RUN_LIVE_LLM:
    raise RuntimeError("Owner-approved live retrieval/LLM check required; keep this branch explicit.")
print("offline_mode=True; no embedding download or LLM call was made")

## Take-away

Retrieval supplies context, structured extraction supplies comparable fields, and synthesis supplies a question for further reading. None of these operations changes the evidential status of the original paper. Keep the PDF filename and page location in any downstream claim.